# Building Footprint Extraction — Accra, Ghana
**NB 02** | Accra Metropolitan District (whole city) | Sentinel-2 10m | SegFormer-B2

This notebook extracts building footprints across the **entire Accra Metropolitan
District** — not a hand-picked sub-area. The AOI is fetched programmatically from
OpenStreetMap's Nominatim API at run time (the official administrative boundary
polygon for "Accra Metropolitan District, Greater Accra Region, Ghana"), so the
study area is the real city boundary, not an assumed bounding box.

**Note on "Accra":** the *Accra Metropolitan District* (governed by the Accra
Metropolitan Assembly, AMA) is the official city-proper boundary — about 60 km²,
population ~284,000 (2021 census). This is smaller than the informal "Greater
Accra" urban agglomeration, which sprawls across ~29 separate districts (Tema,
the Ga districts, Adenta, Ledzokuku, La-Dade Kotopon, etc.) and is not a single
administrative unit. This notebook uses the official AMA boundary since it's the
one unambiguous, citable definition of "Accra" as a city. To instead cover the
wider metro area, change `AOI_QUERY` in the next section to
`"Greater Accra Region, Ghana"` — everything downstream (bbox, clipping, area
stats) adapts automatically since none of it is hard-coded to AMA's specific shape.

Pipeline: fetch real AOI boundary → search → download → validate → preprocess →
clip to true boundary → auto-label (OSM + Microsoft Buildings, for reference /
QA) → SegFormer-B2 inference → validate → postprocess → clip vectors to boundary
→ area statistics → COG + report.

In [1]:
import warnings; warnings.filterwarnings('ignore')
import json
import pathlib

import numpy as np

import pygeovision as pgv
from pygeovision.inference.tiled import TiledInference
from pygeovision.models import get_model

client = pgv.PyGeoVision()
print(client)

17:01:50 INFO [      engine] PyGeoFetch ready


2026-07-23 17:01:50,879 [INFO] pygeofetch.core.engine: PyGeoFetch ready


17:01:50 INFO [      engine] PyGeoFetch ready


2026-07-23 17:01:50,881 [INFO] pygeofetch.core.engine: PyGeoFetch ready
2026-07-23 17:01:50,882 [INFO] pygeovision.data.pgf_bridge: PyGeoFetch v1.0 Python API active — using native search/download
2026-07-23 17:01:50,882 [INFO] pygeovision.data.pgf_bridge: PyGeoFetch v2.0 processing engine detected — using native preprocessing


PyGeoVision(v2.1.6 | pygeofetch=✓ | pgf_v2=✓v2 | ai=✓torch | datasets=503 | models=98 | pipelines=51 | validator=✓ | preprocess=✓ | indices=22 | postprocess=✓ | sar=✓ | labeling=7src | losses=10 | inference=4 | xai=4 | monitoring=3 | cloud=3·aws·azure·gcp | edge=ONNX+Jetson | vlm=CLIP+Moon | few_shot | timeseries | 3D)


## 1. Study Area — fetch the real Accra boundary (no assumed bbox)

We query OpenStreetMap's Nominatim API for the official administrative boundary
polygon, rather than guessing a rectangle. Nominatim's usage policy requires a
descriptive `User-Agent` and a max of ~1 request/second, both respected below.

We then **validate** the result against the known reference area (~60 km²,
Wikipedia/AMA) before trusting it — if a query resolves to the wrong entity
(e.g. a point, or a much larger/smaller region), this check will catch it rather
than silently proceeding with bad data.

In [2]:

AOI_QUERY = "Accra Metropolitan District, Greater Accra Region, Ghana"

DATA_DIR = pathlib.Path('./results/notebook')
DATA_DIR.mkdir(parents=True, exist_ok=True)
AOI_PATH = DATA_DIR / 'accra_boundary.geojson'

aoi = client.boundary(
    AOI_QUERY,
    output_path=str(AOI_PATH),
    reference_area_km2=60.0,
    raise_on_mismatch=False,   # entity confirmed correct — proceed with OSM's actual geometry
)
print(aoi.summary())
if not aoi.validated:
    print("\nNote: proceeding with OSM's boundary as-is despite the area check — "
          "entity match was confirmed correct (category=boundary, exact display_name match); "
          "the deviation reflects OSM-vs-official-statistic divergence, not a wrong match.")

BBOX = aoi.bbox
TARGET_CRS = f"EPSG:{aoi.utm_epsg}"

2026-07-23 17:01:55,808 [WARNING] pygeovision.data.boundary: Fetched boundary area (23.7 km²) deviates 60% from the known reference (60.0 km²) for 'Accra Metropolitan District, Greater Accra Region, Ghana'. Two likely causes: (1) the query resolved to the wrong OSM entity — check osm_type='relation', category='boundary', display_name='Accra Metropolitan District, Greater Accra Region, Ghana' against what you expect; or (2) this is the *right* entity, but OpenStreetMap's community-digitized boundary for it is measurably smaller/larger than the official statistic — a real, known limitation for many regions (especially outside Europe/North America), not necessarily an error. Inspect the geometry yourself (output_path, if given) before deciding whether to proceed with raise_on_mismatch=False or a wider tolerance.


AOI: Accra Metropolitan District, Greater Accra Region, Ghana
  Query               : Accra Metropolitan District, Greater Accra Region, Ghana
  OSM type / class    : relation / boundary
  BBox (WGS84)        : (-0.2555, 5.5228, -0.2009, 5.5994)
  Area (fetched)      : 23.7 km²
  UTM zone (auto)     : EPSG:32630
  Reference area check: 60.0 km² (60.4% deviation) — ✗ FAILED
  ⚠ Fetched boundary area (23.7 km²) deviates 60% from the known reference (60.0 km²) for 'Accra Metropolitan District, Greater Accra Region, Ghana'. Two likely causes: (1) the query resolved to the wrong OSM entity — check osm_type='relation', category='boundary', display_name='Accra Metropolitan District, Greater Accra Region, Ghana' against what you expect; or (2) this is the *right* entity, but OpenStreetMap's community-digitized boundary for it is measurably smaller/larger than the official statistic — a real, known limitation for many regions (especially outside Europe/North America), not necessarily an error. 

## 2. Search & Download — Sentinel-2 over the whole AOI

We search using the AOI's bounding box (satellite catalogues only support
bbox/footprint search, not arbitrary polygons), then check how many scenes are
actually needed to fully cover it — Accra's ~60 km² extent normally fits inside
a single Sentinel-2 MGRS tile, but we verify rather than assume, and mosaic if
more than one tile is required.

In [3]:
DATE_RANGE = ('2024-01-01', '2024-06-30')
PROVIDERS  = ['planetary_computer']
BANDS      = ['B02', 'B03', 'B04', 'B08']

results = client.search(
    bbox            = BBOX,
    date_range      = DATE_RANGE,
    providers       = PROVIDERS,
    cloud_cover_max = 10,
    sort_by         = 'cloud_cover',
    sort_order      = 'asc',
    max_results     = 20,
)
print(f"Found {len(results)} candidate scenes (sorted by cloud cover)")
for r in results[:5]:
    print(f"  {r.provider:<22} {r.datetime[:10]}  cloud={r.cloud_cover:.1f}%  {r.id[:40]}")

2026-07-23 17:02:00,571 [INFO] pygeovision.data.fetch: Searching 1 provider(s) [planetary_computer] | bbox=(-0.255486, 5.522781, -0.200876, 5.599425) | 2024-01-01→2024-06-30 | cloud≤10%



┌──────────────────────────────────────────────────────────────────────┐
│                          🛰  PYGEOFETCH SEARCH                        │
├──────────────────────────────────────────────────────────────────────┤
│  Providers              planetary_computer                                    │
│  BBox                   [-0.255, 5.523, -0.201, 5.599]                        │
│  Date range             2024-01-01  →  2024-06-30                             │
│  Cloud max              10.0%                                                 │
│  Product                any                                                   │
└──────────────────────────────────────────────────────────────────────┘

17:02:01 INFO [planetary_computer] Planetary Computer: 20 results


2026-07-23 17:02:01,835 [INFO] pygeofetch.provider.planetary_computer: Planetary Computer: 20 results
2026-07-23 17:02:01,843 [INFO] pygeovision.data.fetch: Search complete: 20 results


  ✓  planetary_computer              20 scenes   1.3s

  ┌─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┐
  │  SCENE ID                                    DATE        SATELLITE       CLOUD   PRODUCT  POLARISATION  PASS         ORBIT  SCORE  PROVIDER              │
  ├─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┤
  │  S2A_MSIL2A_20240128T101301_R022_T30NYM_202  2024-01-28  Sentinel-2A     0%      —        —            descending   22     0.70   planetary_computer    │
  │  S2A_MSIL2A_20240128T101301_R022_T30NZM_202  2024-01-28  Sentinel-2A     0%      —        —            descending   22     0.70   planetary_computer    │
  │  S2A_MSIL2A_20240131T102251_R065_T30NYM_202  2024-01-31  Sentinel-2A     0%      —        —            descending   65     0.70   planetary_computer  

In [4]:
selected = client.select_covering_scenes(results, BBOX, max_scenes=4)

print(f"Selected {len(selected)} scene(s) to cover the AOI:")
for r in selected:
    print(f"  {r.datetime[:10]}  cloud={r.cloud_cover:.1f}%  {r.id[:40]}")

downloads = client.download(
    selected,
    output_dir   = str(DATA_DIR),
    bands        = BANDS,
    post_process = [f'reproject:{TARGET_CRS}', 'cog'],
)
ok_downloads = [d for d in downloads if d.success]
print(f"\nDownloaded {len(ok_downloads)}/{len(downloads)} scene(s) successfully")
for d in ok_downloads:
    print(f"  {d.path}  ({d.bytes_downloaded/1024/1024:.1f} MB)")

Selected 4 scene(s) to cover the AOI:
  2024-01-28  cloud=0.0%  S2A_MSIL2A_20240128T101301_R022_T30NYM_2
  2024-01-28  cloud=0.3%  S2A_MSIL2A_20240128T101301_R022_T30NZM_2
  2024-01-31  cloud=0.3%  S2A_MSIL2A_20240131T102251_R065_T30NYM_2
  2024-01-31  cloud=0.6%  S2A_MSIL2A_20240131T102251_R065_T30NZM_2

  📡 Downloading 4 scene(s) via PyGeoFetch from planetary_computer
  📁 Output: results/notebook
  🔧 Post-process: reproject:EPSG:32630 → cog
  ⚡ Parallel downloads: 4

  ⠋  Downloading 4 scene(s)…  
  ┌──────────────────────────────────────────────────────────────────────┐
  │  ⬇  DOWNLOADING  4 scenes  → results/notebook                        │
  └──────────────────────────────────────────────────────────────────────┘

17:02:06 INFO [  downloader] Downloading S2A_MSIL2A_20240128T101301_R022_T30NYM_20240128T151508


2026-07-23 17:02:06,515 [INFO] pygeofetch.core.downloader: Downloading S2A_MSIL2A_20240128T101301_R022_T30NYM_20240128T151508


17:02:06 INFO [  downloader] Downloading S2A_MSIL2A_20240128T101301_R022_T30NZM_20240128T151522
17:02:06 INFO [  downloader] Downloading S2A_MSIL2A_20240131T102251_R065_T30NZM_20240131T162149
17:02:06 INFO [  downloader] Downloading S2A_MSIL2A_20240131T102251_R065_T30NYM_20240131T162209


2026-07-23 17:02:06,515 [INFO] pygeofetch.core.downloader: Downloading S2A_MSIL2A_20240128T101301_R022_T30NZM_20240128T151522
2026-07-23 17:02:06,517 [INFO] pygeofetch.core.downloader: Downloading S2A_MSIL2A_20240131T102251_R065_T30NZM_20240131T162149
2026-07-23 17:02:06,517 [INFO] pygeofetch.core.downloader: Downloading S2A_MSIL2A_20240131T102251_R065_T30NYM_20240131T162209


  ⠧  Downloading 4 scene(s)…  17:02:07 INFO [planetary_computer] Downloading 15 asset(s) for S2A_MSIL2A_20240128T101301_R022_T30NYM_20240128T151508: ['AOT', 'B01', 'B02', 'B03', 'B04', 'B05', 'B06', 'B07', 'B08', 'B09', 'B11', 'B12', 'B8A', 'SCL', 'WVP']


...

  ⠏  Downloading 4 scene(s)…  17:38:32 INFO [planetary_computer]   ✓ T30NYM_20240131T102251_WVP_10m.tif (78.6 MB)


2026-07-23 17:38:32,784 [INFO] pygeofetch.provider.planetary_computer:   ✓ T30NYM_20240131T102251_WVP_10m.tif (78.6 MB)


  ⠧  Downloading 4 scene(s)…  17:41:08 INFO [  downloader]   ✓ S2A_MSIL2A_20240131T102251_R065_T30NYM_202401   1239 MB  2185.3s


2026-07-23 17:41:08,442 [INFO] pygeofetch.core.downloader:   ✓ S2A_MSIL2A_20240131T102251_R065_T30NYM_202401   1239 MB  2185.3s


  ✓  [4/4]                                                        1.21 GB       2341.9s

  ✓  All 4 scenes downloaded   total time: 2341.9s

  [████████████████████████████████████████] 4/4 (100%) | 2728.5 MB

  ──────────────────────────────────────────────────
  ✅ Download complete: 4/4 scenes
  📦 Total size: 2728.5 MB
  ⏱️  Duration: 2341.9 seconds (39.0 minutes)
  ──────────────────────────────────────────────────


Downloaded 4/4 scene(s) successfully
  results/notebook/planetary_computer/T30NYM_20240128T101301_AOT_10m_EPSG_32630_cog.tif  (124.2 MB)
  results/notebook/planetary_computer/T30NZM_20240128T101301_AOT_10m_EPSG_32630_cog.tif  (1128.0 MB)
  results/notebook/planetary_computer/T30NYM_20240131T102251_AOT_10m_EPSG_32630_cog.tif  (1239.3 MB)
  results/notebook/planetary_computer/T30NZM_20240131T102251_AOT_10m_EPSG_32630_cog.tif  (236.9 MB)


## 3. Mosaic (if needed) & validate raw data

In [5]:
scl_cands = list(DATA_DIR.rglob('*SCL*.tif')) + list(DATA_DIR.rglob('*scl*.tif'))
scl_path  = str(scl_cands[0]) if scl_cands else None

scene_path = None
if ok_downloads:
    scene_path = client.preprocess.mosaic(
        [d.path for d in ok_downloads],
        output_path=str(DATA_DIR / 'accra_raw_mosaic.tif'),
    )
    print(f"Working raster ({len(ok_downloads)} scene(s)): {scene_path}")
else:
    print("No scenes downloaded — check provider availability / date range")

2026-07-23 17:49:14,888 [INFO] pygeovision.preprocess.core: Mosaicked 4 rasters → results/notebook/accra_raw_mosaic.tif (20982x10980 px)


Working raster (4 scene(s)): results/notebook/accra_raw_mosaic.tif


In [6]:
if scene_path:
    raw_report = client.validator.validate(
        str(scene_path),
        required_bands = len(BANDS),
        value_range    = (0, 65535),
    )
    print("Raw data validation:")
    print(raw_report.summary())
    if not raw_report.passed:
        print("WARNING: Issues found — preprocessing will auto-fix")

2026-07-23 17:50:17,845 [WARNING] pygeovision.data.validator: Auto-fixed 1 validation error(s) for 'results/notebook/accra_raw_mosaic.tif'


Raw data validation:
ValidationReport: results/notebook/accra_raw_mosaic.tif
  Status : ✗ FAILED
  Errors : 1
  Warnings: 1
  Stats  : {'bands': 1, 'height': 10980, 'width': 20982, 'dtype': 'float32', 'value_min': 668.4979248046875, 'value_max': 1188.0, 'value_mean': 1053.0970458984375, 'value_std': 96.14968872070312, 'nan_pct': 0.0, 'zero_pct': 0.0}
  [✗ ERROR] check_bands: Too few bands: have 1, need ≥ 4  → Stack additional bands or use a model expecting 1 channels.
  [⚠ WARNING] check_outliers: Band 0: 4 outlier(s) beyond 4.0σ  → clipped to [668.50, 1437.70]


## 4. Preprocess & clip to the exact Accra boundary

`prepare_for_ai` uses the AOI **bbox** to build the model-ready stack (band
selection, cloud masking, normalisation). We then clip precisely to the real
boundary **polygon** with `clip_to_polygon`, so everything downstream — the
prediction, the vectorised buildings, the area statistics — reflects the true
city shape, not the rectangle around it.

In [ ]:
PREPROCESSED   = DATA_DIR / 'accra_preprocessed.tif'
PREPROCESSED_CLIPPED = DATA_DIR / 'accra_preprocessed_clipped.tif'

if scene_path:
    ready = client.prepare_for_ai(
        str(scene_path),
        stack_bands  = BANDS,
        bbox         = BBOX,
        scl_path         = scl_path,
        scl_keep_classes = [4, 5, 6],
        normalise    = 'scale_factor',
        scale_factor = 10000.0,
        model_type   = 'segmentation',
        output_path  = str(PREPROCESSED),
    )
    print("Preprocessing steps :", ready['steps'])
    print("Output shape (C,H,W):", ready['shape'])
    print("Resolution          :", ready['resolution_m'], "m")
    print("Validation          :", "PASSED" if ready['report'] and ready['report'].passed else "FIXED (auto)")

    clipped_path = client.preprocess.clip_to_polygon(
        str(PREPROCESSED),
        geojson=str(AOI_PATH),
        output_path=str(PREPROCESSED_CLIPPED),
        crop=True,
    )
    print(f"Clipped to true AOI boundary → {clipped_path}")
else:
    ready = None
    clipped_path = None
    print("No scene available — cannot preprocess")

## 5. Reference labels — OSM + Microsoft Buildings (for QA, not training)

Two independent open building-footprint sources, fused, give us a reference
layer to sanity-check the model's predictions against later (§8) — a basic
but genuine accuracy signal, not just a visual check. Both sources cover the
*whole* AOI bbox, matching the study area rather than a sample corner of it.

In [ ]:
from pygeovision.labeling.pipeline import AutoLabelPipeline

LABELS_DIR = DATA_DIR / 'labels'
LABELS_DIR.mkdir(exist_ok=True)

ref_pipeline = AutoLabelPipeline(
    sources=["osm", "microsoft_buildings"],
    fusion="union",
)
ref_result = ref_pipeline.run(bbox=BBOX, output_dir=str(LABELS_DIR))
print(json.dumps({k: v for k, v in ref_result.items() if k != 'partial_results'}, indent=2, default=str))

## 6. Inference — SegFormer-B2, tiled with Gaussian blending

In [ ]:
PREDICTION = DATA_DIR / 'buildings_raw.tif'

infer_input = clipped_path or (str(PREPROCESSED) if PREPROCESSED.exists() else None)

if infer_input:
    model = get_model('segformer-b2', num_classes=2, pretrained=True)
    inf   = TiledInference(model, chip_size=512, overlap=64, blend_mode='gaussian')
    res   = inf.infer(infer_input, str(PREDICTION))
    print(f"Inference: {res.get('n_chips')} chips  {res.get('duration_seconds', 0):.1f}s  "
          f"→ {PREDICTION}")
else:
    print("No preprocessed raster available — skipping inference")

In [ ]:
if PREDICTION.exists():
    pred_report = client.validator.validate(str(PREDICTION))
    print("Prediction validation:", "PASSED" if pred_report.passed else f"FIXED — {pred_report.errors}")
    print(pred_report.summary())

## 7. Postprocess — sieve, regularise, vectorise, clip to true boundary

The raster was already clipped to Accra's real shape before inference, but
tiled inference can still leave partial/edge artifacts right at the boundary.
We clip the final **vector** output to the exact polygon too, so no building
polygon extends past the true city limit.

In [ ]:
SIEVED       = DATA_DIR / 'sieved.tif'
VECTOR       = DATA_DIR / 'output.geojson'
VECTOR_REG   = DATA_DIR / 'output_regularised.geojson'
VECTOR_FINAL = DATA_DIR / 'accra_buildings_final.geojson'
COG_OUT      = DATA_DIR / 'output_cog.tif'

if PREDICTION.exists():
    client.postprocess.sieve_filter(str(PREDICTION), min_pixels=25, output_path=str(SIEVED))
    client.postprocess.vectorise(
        str(SIEVED), str(VECTOR),
        target_class=1, min_area_m2=25.0, simplify_tolerance=0.3,
    )
    client.postprocess.regularise_buildings(str(VECTOR), str(VECTOR_REG))

    # Final precise clip to the true AOI polygon (drops/trims anything outside it)
    import geopandas as gpd
    buildings_gdf = gpd.read_file(VECTOR_REG)
    aoi_gdf = gpd.GeoDataFrame(geometry=[aoi_geom_wgs84], crs="EPSG:4326").to_crs(buildings_gdf.crs)
    clipped_buildings = gpd.clip(buildings_gdf, aoi_gdf)
    clipped_buildings.to_file(VECTOR_FINAL, driver="GeoJSON")

    client.postprocess.to_cog(str(SIEVED), str(COG_OUT))
    print(f"Buildings before boundary clip : {len(buildings_gdf)}")
    print(f"Buildings after boundary clip  : {len(clipped_buildings)}")
    print(f"Final output → {VECTOR_FINAL}")
else:
    print("No prediction available — skipping postprocess")

## 8. Area statistics & reference comparison

In [ ]:
if PREDICTION.exists():
    stats = client.postprocess.class_statistics(str(SIEVED))
    print(f"{'Class':<8} {'Area (ha)':>12} {'Area (km²)':>12} {'% of AOI':>10}")
    print("-" * 46)
    for cls, info in stats.items():
        print(f"{cls:<8} {info['area_ha']:>12.1f} {info['area_km2']:>12.3f} {info['pct']:>9.1f}%")

    building_area_km2 = stats.get(1, {}).get("area_km2", 0.0)
    print(f"\nModel-predicted building coverage : {building_area_km2:.2f} km² "
          f"({100 * building_area_km2 / fetched_area_km2:.1f}% of AOI)")
    print(f"Reference (OSM ∪ MS Buildings)     : "
          f"{ref_result.get('n_features', 'n/a')} footprints "
          f"({ref_result.get('output_path', 'n/a')})")

## 9. Report

In [ ]:
if PREDICTION.exists():
    client.postprocess.generate_report(str(SIEVED), str(DATA_DIR / 'accra_report.html'))
    print(f"Report → {DATA_DIR / 'accra_report.html'}")

**NB02 — Building Extraction (Accra, whole city)** complete.

- **Study area**: Accra Metropolitan District, Ghana — fetched from OSM Nominatim
  at run time (not an assumed bounding box), boundary area cross-validated
  against the official ~60 km² AMA figure.
- **Sensor**: Sentinel-2 10m, mosaicked across scenes if the AOI spans more
  than one tile.
- **Model**: SegFormer-B2, tiled Gaussian-blended inference.
- **Reference labels**: OSM + Microsoft Global ML Buildings (union), for
  independent QA against the model's predictions.
- **Pipeline**: fetch real AOI → search → download → mosaic-if-needed →
  validate_raw → preprocess → clip to true boundary → auto-label (reference)
  → model inference → validate_prediction → postprocess → clip vectors to
  true boundary → area statistics vs. reference → COG + HTML report.